# BERT Sentiment Analysis
**Model:** `bert-base-uncased` (fine-tuned for binary sentiment classification)

**Dataset:** `merged_sentiment.csv` (50,000 IMDB reviews + 1,600,000 tweets)

**Training sample:** 100,000 rows — all 50,000 IMDB reviews + 50,000 tweets (balanced: 50k positive / 50k negative)

---
### Instructions
1. Go to **Runtime → Change runtime type → T4 GPU** before running.
2. Make sure `merged_sentiment.csv` is uploaded to your Google Drive.
3. Run all cells in order.
4. Download `bert_sentiment_model.zip` at the end and place the extracted folder in your `data/` directory.

In [ ]:
# Check GPU
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))

In [ ]:
# Install HuggingFace Transformers
!pip install -q transformers accelerate

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH if merged_sentiment.csv is inside a folder ──
CSV_PATH = '/content/drive/MyDrive/merged_sentiment.csv'

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
SAMPLE_SIZE  = 100_000   # total rows (50k positive + 50k negative)
IMDB_TOTAL   = 50_000    # all IMDB rows always included
MAX_LEN      = 128       # max tokens per text
BATCH_SIZE   = 32        # reduce to 16 if you get out-of-memory errors
EPOCHS       = 3         # 3 epochs is standard for BERT fine-tuning
LEARNING_RATE = 2e-5
MODEL_NAME   = 'bert-base-uncased'
# ─────────────────────────────────────────────────────────────────

In [ ]:
# Load merged_sentiment.csv
print('Loading data ...')
df = pd.read_csv(CSV_PATH, dtype={'label': int})
print(f'Total rows: {len(df):,}')
print(df['label'].value_counts())

In [ ]:
# Sample: all IMDB rows + tweets to fill remaining slots
imdb_df  = df.iloc[:IMDB_TOTAL]
tweet_df = df.iloc[IMDB_TOTAL:]

half             = SAMPLE_SIZE // 2
tweet_pos_slots  = half - int((imdb_df['label'] == 1).sum())
tweet_neg_slots  = half - int((imdb_df['label'] == 0).sum())

tweet_pos = tweet_df[tweet_df['label'] == 1].sample(n=tweet_pos_slots, random_state=SEED)
tweet_neg = tweet_df[tweet_df['label'] == 0].sample(n=tweet_neg_slots, random_state=SEED)

sample_df = pd.concat([imdb_df, tweet_pos, tweet_neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample_df['text'] = sample_df['text'].astype(str)

print(f'Sample size: {len(sample_df):,}')
print(sample_df['label'].value_counts())

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    sample_df['text'].tolist(),
    sample_df['label'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=sample_df['label'].tolist()
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

In [ ]:
# Load BERT tokenizer
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

In [ ]:
# PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'token_type_ids': self.encodings['token_type_ids'][idx],
            'labels':         self.labels[idx],
        }

print('Tokenising training set ...')
train_dataset = SentimentDataset(X_train, y_train, tokenizer, MAX_LEN)
print('Tokenising test set ...')
test_dataset  = SentimentDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}   Test batches: {len(test_loader)}')

In [ ]:
# Load pre-trained BERT model with classification head
print(f'Loading model: {MODEL_NAME} ...')
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)
print('Model loaded.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total training steps : {total_steps:,}')
print(f'Warmup steps         : {warmup_steps:,}')

In [ ]:
# Training loop
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch['token_type_ids'].to(DEVICE)
            labels         = batch['labels']
            outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)
    return acc, f1


best_f1    = 0
train_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch['token_type_ids'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if (step + 1) % 200 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/(step+1):.4f}')

    avg_loss = total_loss / len(train_loader)
    val_acc, val_f1 = evaluate(model, test_loader)
    train_history.append({'epoch': epoch, 'loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})

    print(f'\nEpoch {epoch}/{EPOCHS} — Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.1%} | Val F1: {val_f1:.1%}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained('bert_sentiment_model')
        tokenizer.save_pretrained('bert_sentiment_model')
        print(f'  ✓ Best model saved (F1 = {best_f1:.1%})')
    print()

In [ ]:
# Final evaluation on test set
from transformers import BertForSequenceClassification as BertCls
best_model = BertCls.from_pretrained('bert_sentiment_model').to(DEVICE)

best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch['token_type_ids'].to(DEVICE)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask,
                             token_type_ids=token_type_ids)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())

print('=== Final Test Results ===')
print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

epochs_range = [h['epoch'] for h in train_history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, [h['loss']    for h in train_history], marker='o')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')

ax2.plot(epochs_range, [h['val_acc'] for h in train_history], marker='o', label='Accuracy')
ax2.plot(epochs_range, [h['val_f1']  for h in train_history], marker='s', label='F1')
ax2.set_title('Validation Metrics')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Zip and download the model folder
import shutil
shutil.make_archive('bert_sentiment_model', 'zip', 'bert_sentiment_model')
print('Zipped: bert_sentiment_model.zip')

from google.colab import files
files.download('bert_sentiment_model.zip')

## Next steps
1. Extract `bert_sentiment_model.zip`
2. Place the extracted folder inside the `data/` directory of your thesis project:
```
data/bert_sentiment_model/config.json
data/bert_sentiment_model/model.safetensors
data/bert_sentiment_model/tokenizer.json
data/bert_sentiment_model/vocab.txt
...
```
3. The FastAPI backend will then load and serve BERT predictions alongside the other models.